# Notebook 8 – Text Feature Engineering

## Learning Objectives
Convert raw text into Machine Learning features. Create text length, character count, word count, sentence count, digit count, special-character count, uppercase count, lowercase count, keyword count, Bag of Words, TF-IDF, and N-Grams.

## 1. What is Text Feature Engineering?
Text feature engineering converts unstructured text into numeric or structured information that a Machine Learning model can process.

**Example:** A product review can be represented using word counts, keyword indicators, or TF-IDF values.

## 2. Sample Text Dataset

In [1]:
import pandas as pd
import numpy as np

reviews = pd.DataFrame({
    "review_id":[1,2,3,4,5],
    "review_text":[
        "The product is excellent! Delivery was fast.",
        "Very poor quality, not happy with the product.",
        "Good value for money. I would buy again.",
        "Delivery was delayed by 3 days!!!",
        "EXCELLENT quality and great service 10/10."
    ]
})
reviews

,review_id,review_text
0,1,The product is excellent! Delivery was fast.
1,2,"Very poor quality, not happy with the product."
2,3,Good value for money. I would buy again.
3,4,Delivery was delayed by 3 days!!!
4,5,EXCELLENT quality and great service 10/10.


**Output Explanation:** Each row contains one product review. The text needs to be represented as measurable features for many traditional ML algorithms.

## 3. Text Length
Text length measures the number of characters, including spaces and punctuation.

In [2]:
reviews["text_length"] = reviews["review_text"].str.len()
reviews[["review_text","text_length"]]

,review_text,text_length
0,The product is excellent! Delivery was fast.,44
1,"Very poor quality, not happy with the product.",46
2,Good value for money. I would buy again.,40
3,Delivery was delayed by 3 days!!!,33
4,EXCELLENT quality and great service 10/10.,42


**Output Explanation:** This counts every character. Longer reviews may contain more detail, but length does not represent meaning.

## 4. Character Count
Character count can be measured with or without spaces. This example excludes whitespace.

In [3]:
reviews["character_count_no_spaces"] = reviews["review_text"].str.replace(r"\s+","",regex=True).str.len()
reviews[["review_text","character_count_no_spaces"]]

,review_text,character_count_no_spaces
0,The product is excellent! Delivery was fast.,38
1,"Very poor quality, not happy with the product.",39
2,Good value for money. I would buy again.,33
3,Delivery was delayed by 3 days!!!,28
4,EXCELLENT quality and great service 10/10.,37


**Output Explanation:** Whitespace is removed before counting. This differs from total text length.

## 5. Word Count
A simple word count can be calculated by splitting text on whitespace.

In [4]:
reviews["word_count"] = reviews["review_text"].str.split().str.len()
reviews[["review_text","word_count"]]

,review_text,word_count
0,The product is excellent! Delivery was fast.,7
1,"Very poor quality, not happy with the product.",8
2,Good value for money. I would buy again.,8
3,Delivery was delayed by 3 days!!!,6
4,EXCELLENT quality and great service 10/10.,6


**Output Explanation:** This counts whitespace-separated pieces. Punctuation may remain attached to words.

## 6. Sentence Count
A simple approach counts segments separated by periods, exclamation marks, and question marks. It is an approximation.

In [5]:
reviews["sentence_count"] = reviews["review_text"].str.split(r"[.!?]+").apply(
    lambda parts: sum(bool(part.strip()) for part in parts)
)
reviews[["review_text","sentence_count"]]

,review_text,sentence_count
0,The product is excellent! Delivery was fast.,2
1,"Very poor quality, not happy with the product.",1
2,Good value for money. I would buy again.,2
3,Delivery was delayed by 3 days!!!,1
4,EXCELLENT quality and great service 10/10.,1


**Output Explanation:** This basic rule may not handle abbreviations or complex punctuation correctly.

## 7. Number of Digits
Count numeric characters in the text.

In [6]:
reviews["digit_count"] = reviews["review_text"].str.count(r"\d")
reviews[["review_text","digit_count"]]

,review_text,digit_count
0,The product is excellent! Delivery was fast.,0
1,"Very poor quality, not happy with the product.",0
2,Good value for money. I would buy again.,0
3,Delivery was delayed by 3 days!!!,1
4,EXCELLENT quality and great service 10/10.,4


**Output Explanation:** The feature counts each digit from 0 to 9, which can be useful for text containing quantities or product codes.

## 8. Number of Special Characters
Count characters outside letters, digits, and whitespace. The definition depends on the application.

In [7]:
reviews["special_character_count"] = reviews["review_text"].str.count(r"[^A-Za-z0-9\s]")
reviews[["review_text","special_character_count"]]

,review_text,special_character_count
0,The product is excellent! Delivery was fast.,2
1,"Very poor quality, not happy with the product.",2
2,Good value for money. I would buy again.,2
3,Delivery was delayed by 3 days!!!,3
4,EXCELLENT quality and great service 10/10.,2


**Output Explanation:** This counts punctuation and symbols. This pattern is ASCII-oriented; multilingual text may require Unicode-aware handling.

## 9. Uppercase Count
Count uppercase alphabetic characters.

In [8]:
reviews["uppercase_count"] = reviews["review_text"].apply(
    lambda text: sum(ch.isupper() for ch in text)
)
reviews[["review_text","uppercase_count"]]

,review_text,uppercase_count
0,The product is excellent! Delivery was fast.,2
1,"Very poor quality, not happy with the product.",1
2,Good value for money. I would buy again.,2
3,Delivery was delayed by 3 days!!!,1
4,EXCELLENT quality and great service 10/10.,9


**Output Explanation:** Uppercase use may represent emphasis, acronyms, or shouting, depending on context.

## 10. Lowercase Count
Count lowercase alphabetic characters.

In [9]:
reviews["lowercase_count"] = reviews["review_text"].apply(
    lambda text: sum(ch.islower() for ch in text)
)
reviews[["review_text","lowercase_count"]]

,review_text,lowercase_count
0,The product is excellent! Delivery was fast.,34
1,"Very poor quality, not happy with the product.",36
2,Good value for money. I would buy again.,29
3,Delivery was delayed by 3 days!!!,23
4,EXCELLENT quality and great service 10/10.,22


**Output Explanation:** This describes the amount of lowercase text and can be considered with uppercase count.

## 11. Keyword Count
Keyword count measures occurrences of selected words. The keyword list should match the domain and task.

In [10]:
positive_keywords = ["excellent","good","great","fast","happy"]
negative_keywords = ["poor","bad","delayed","not"]

def count_keywords(text, keywords):
    text = text.lower()
    return sum(text.count(word) for word in keywords)

reviews["positive_keyword_count"] = reviews["review_text"].apply(
    lambda text: count_keywords(text, positive_keywords)
)
reviews["negative_keyword_count"] = reviews["review_text"].apply(
    lambda text: count_keywords(text, negative_keywords)
)
reviews[["review_text","positive_keyword_count","negative_keyword_count"]]

,review_text,positive_keyword_count,negative_keyword_count
0,The product is excellent! Delivery was fast.,2,0
1,"Very poor quality, not happy with the product.",1,2
2,Good value for money. I would buy again.,1,0
3,Delivery was delayed by 3 days!!!,0,1
4,EXCELLENT quality and great service 10/10.,2,0


**Output Explanation:** This counts listed terms after lowercasing. Substring matching may count parts of longer words; token-based matching may be preferable. Keyword counts are not a complete sentiment model.

## 12. Bag of Words (BoW)
Bag of Words converts text into a matrix of token counts. Each column represents a vocabulary term.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

documents = reviews["review_text"].fillna("")
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(documents)
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)
bow_df

,10,again,and,buy,by,days,delayed,delivery,excellent,fast,...,poor,product,quality,service,the,value,very,was,with,would
0,0,0,0,0,0,0,0,1,1,1,...,0,1,0,0,1,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,0,1,0,1,0
2,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
3,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
4,2,0,1,0,0,0,0,0,1,0,...,0,0,1,1,0,0,0,0,0,0


**Output Explanation:** Each cell contains a word count for one review. BoW captures term frequency but does not preserve word order or full context.

## 13. TF-IDF
TF-IDF (Term Frequency–Inverse Document Frequency) weights terms based on their frequency in a document and how common they are across the document collection.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)
tfidf_df.round(3)

,10,again,and,buy,by,days,delayed,delivery,excellent,fast,...,poor,product,quality,service,the,value,very,was,with,would
0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.352,0.352,0.436,...,0.000,0.352,0.000,0.000,0.352,0.000,0.000,0.352,0.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.379,0.306,0.306,0.000,0.306,0.000,0.379,0.000,0.379,0.000
2,0.000,0.378,0.000,0.378,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.378,0.000,0.000,0.000,0.378
3,0.000,0.000,0.000,0.000,0.482,0.482,0.482,0.389,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.389,0.000,0.000
4,0.694,0.000,0.347,0.000,0.000,0.000,0.000,0.000,0.280,0.000,...,0.000,0.000,0.280,0.347,0.000,0.000,0.000,0.000,0.000,0.000


**Output Explanation:** Values are TF-IDF weights rather than raw counts. Fit the vectorizer on training text only, then transform validation and test text.

## 14. N-Grams
N-grams are sequences of adjacent tokens. Unigrams contain one word, bigrams two consecutive words, and trigrams three.

In [13]:
ngram_vectorizer = CountVectorizer(ngram_range=(1,2))
ngram_matrix = ngram_vectorizer.fit_transform(documents)
ngram_df = pd.DataFrame(
    ngram_matrix.toarray(),
    columns=ngram_vectorizer.get_feature_names_out()
)
ngram_df

,10,10 10,again,and,and great,buy,buy again,by,by days,days,...,value for,very,very poor,was,was delayed,was fast,with,with the,would,would buy
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,1,1,0,0
2,0,0,1,0,0,1,1,0,0,0,...,1,0,0,0,0,0,0,0,1,1
3,0,0,0,0,0,0,0,1,1,1,...,0,0,0,1,1,0,0,0,0,0
4,2,1,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Output Explanation:** The vocabulary contains single words and two-word sequences. Bigrams preserve some local word order but increase feature count.